# Advanced Pattern – Self-Querying Retrievers

In standard applications, users write natural language queries, but application developers have to hardcode metadata filters or force users to manually select choices from UI dropdown menus (e.g., “Filter by Year > 2024”).

A Self-Querying Retriever eliminates this friction by leveraging an LLM to dynamically inspect the user's conversational prompt, separate the semantic search intent from the metadata constraints, and build a structured database query on the fly.

## 1. How Self-Querying Works Under the Hood
When a user submits a natural language sentence—such as:

"Find me sci-fi movies about space exploration released after 2010 directed by Christopher Nolan."

A self-querying pipeline splits the prompt into two independent parts:

The Semantic Search Query: "space exploration" (used for vector embedding similarity lookups).

The Metadata Filter Rules: genre == "science fiction" AND year > 2010 AND director == "Christopher Nolan" (converted into the vector store's native filter payload syntax).

# 2. Implementation Pattern (LangChain + Vector Store)
To implement a self-querying retriever, you must explicitly provide the LLM with metadata schema definitions using AttributeInfo so it knows what fields exist in your database.

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_chroma import Chroma
from langchain.chains.query_constructor.base import AttributeInfo
from langchain.retrievers.self_query.base import SelfQueryRetriever

# 1. Define sample documents with rich metadata
docs = [
    Document(
        page_content="A team of explorers travel through a wormhole in space to ensure humanity's survival.",
        metadata={"year": 2014, "director": "Christopher Nolan", "rating": 8.6, "genre": "science fiction"}
    ),
    Document(
        page_content="A thief who steals corporate secrets through the use of dream-sharing technology.",
        metadata={"year": 2010, "director": "Christopher Nolan", "rating": 8.8, "genre": "action"}
    ),
]

# 2. Initialize vector store
embeddings = OpenAIEmbeddings()
vector_store = Chroma.from_documents(docs, embeddings)

# 3. Define the metadata schema so the LLM understands available fields
metadata_field_info = [
    AttributeInfo(
        name="genre",
        description="The genre of the movie (e.g., science fiction, action, drama)",
        type="string",
    ),
    AttributeInfo(
        name="year",
        description="The year the movie was released",
        type="integer",
    ),
    AttributeInfo(
        name="director",
        description="The name of the movie director",
        type="string",
    ),
    AttributeInfo(
        name="rating",
        description="The IMDb rating of the movie on a scale of 1 to 10",
        type="float",
    ),
]

# 4. Initialize the LLM responsible for query construction
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 5. Create the Self-Querying Retriever
document_content_description = "Brief summary of a movie"
retriever = SelfQueryRetriever.from_llm(
    llm=llm,
    vectorstore=vector_store,
    document_contents=document_content_description,
    metadata_field_info=metadata_field_info,
    verbose=True
)

# 6. Execute a natural language query with implicit filters
# The LLM parses this into vector text ("wormhole space") + filter (year > 2012 AND director == Christopher Nolan)
relevant_docs = retriever.invoke("What is that sci-fi movie about a wormhole after 2012?")

for doc in relevant_docs:
    print(f"Content: {doc.page_content} \nMetadata: {doc.metadata}\n")

# 3. Production Considerations & Edge Cases
Schema Documentation Quality: The descriptions you write inside AttributeInfo act as prompt instructions for the LLM. If your description is vague (e.g., description="date"), the LLM will struggle to map natural language phrases like "last year" or "post-2020" to the correct filter keys.

Latency Overhead: Because an extra LLM call is required to parse the query string before executing the database search, expect a slight latency penalty (typically 200–500ms depending on the model).

Hallucinated Filters: If a user asks for something outside your schema (e.g., "movies starring Tom Cruise" when actor is not indexed), a weak LLM might attempt to invent a filter key, causing database query syntax errors. Using strict output parsers and validation safeguards mitigates this risk.